# R8 — ai_query batch adjudication

The client wants an LLM first pass on claim adjudication, but nothing unchecked: first a **deterministic rule-based verifier in SQL** (claims data + plan terms) decides every probe claim; then the **same claims are adjudicated by an LLM in SQL** (`ai_query`, fixed JSON reply shape, temperature 0, reasoning kept, worked examples on made-up claims in the prompt). The LLM never sees the verifier's answers. It is accepted only if it **agrees with the verifier on at least 90% of claims, including all four trap claims**.

Cells:
1. Load `adjudication_probe_claims.csv` → `claims.genai.adjudication_probes` (claim_id STRING).
2. SQL verifier → `claims.genai.adjudication_verifier` (7 columns, one row per probe claim).
3. Batch LLM adjudication in SQL (`ai_query` + `responseFormat`, parsed with `from_json`) → `claims.genai.adjudications` (7 columns + reasoning).
4. Agreement check (>=90% + all 4 trap claims) and the screenshot directions for `evidence/screenshots/r8_adjudications.png`.
5. Independent verification: hand-recomputed spot checks, exact table types, LLM JSON validity, reasoning-length rule.

Rules (from the Definitions sheet): initial waiting passes when days from first policy start >= plan initial waiting days; PED waiting passes when not pre-existing or tenure (completed months from first policy start) >= plan PED waiting months; payable = waiting_period_ok; no exclusions; network_at_admission = network flag on the admission date; room rent deduction = 0 if rent <= cap else (1 - cap/rent)*100 rounded to 2 decimals; co-pay = plan co-pay pct + 20 if out of network on admission; reason code first-match INITIAL_WAITING -> PED_WAITING -> PAYABLE_WITH_DEDUCTION -> PAYABLE_IN_FULL. Agreement: payable, waiting_period_ok, network_at_admission, copay_pct, reason_code equal AND room_rent_deduction_pct within 1.0 point.

In [0]:

# Step 1 - the probe claims -> claims.genai.adjudication_probes (claim_id STRING).

ids = (spark.read.text("/Volumes/claims/raw/landing/genai/adjudication_probe_claims.csv")
       .filter("value <> 'claim_id'")
       .selectExpr("value AS claim_id"))
ids.createOrReplaceTempView("probe_ids")
print("probe claims:", ids.count())

ids.write.mode("overwrite").saveAsTable("claims.genai.adjudication_probes")
print(spark.sql("DESCRIBE TABLE claims.genai.adjudication_probes").filter("col_name = 'claim_id'").collect()[0])

# verifier input facts (gold table = claims joined to the policy version at admission + network status)
spark.sql("""
CREATE OR REPLACE TEMP VIEW probe_claims AS
SELECT p.claim_id,
       e.plan_code_at_admission AS plan_code,
       e.admission_date, e.policy_start_date, e.policy_tenure_months,
       e.is_pre_existing, e.room_rent_per_day, e.is_network_at_admission,
       CAST(pl.initial_waiting_days AS INT)     AS initial_waiting_days,
       CAST(pl.ped_waiting_months AS INT)       AS ped_waiting_months,
       CAST(pl.copay_pct AS INT)                AS plan_copay_pct,
       CAST(pl.room_rent_cap_per_day AS DOUBLE) AS room_rent_cap
FROM claims.genai.adjudication_probes p
JOIN claims.gold.claims_enriched e ON e.claim_id = p.claim_id
JOIN claims.raw.plans pl ON pl.plan_code = e.plan_code_at_admission
""")
assert spark.sql("SELECT count(*) c FROM probe_claims").collect()[0].c == 20
print("probe_claims view ready (20 rows, plan terms joined)")

In [0]:
# Step 2 - the deterministic rule-based verifier, pure SQL -> claims.genai.adjudication_verifier.

spark.sql("""
CREATE OR REPLACE TABLE claims.genai.adjudication_verifier AS
SELECT
  CAST(claim_id AS STRING)                                                                 AS claim_id,
  CAST(waiting_period_ok AS BOOLEAN)                                                       AS payable,
  CAST(waiting_period_ok AS BOOLEAN)                                                       AS waiting_period_ok,
  CAST(network_at_admission AS BOOLEAN)                                                    AS network_at_admission,
  CAST(room_rent_deduction_pct AS DOUBLE)                                                  AS room_rent_deduction_pct,
  CAST(copay_pct AS INT)                                                                   AS copay_pct,
  CAST(reason_code AS STRING)                                                              AS reason_code
FROM (
  SELECT claim_id,
         (datediff(admission_date, policy_start_date) >= initial_waiting_days
          AND (NOT is_pre_existing OR policy_tenure_months >= ped_waiting_months))          AS waiting_period_ok,
         is_network_at_admission                                                            AS network_at_admission,
         CASE WHEN room_rent_per_day <= room_rent_cap THEN 0.0
              ELSE round((1 - room_rent_cap / room_rent_per_day) * 100, 2) END              AS room_rent_deduction_pct,
         plan_copay_pct + CASE WHEN is_network_at_admission THEN 0 ELSE 20 END              AS copay_pct,
         CASE WHEN NOT (datediff(admission_date, policy_start_date) >= initial_waiting_days) THEN 'INITIAL_WAITING'
              WHEN NOT (NOT is_pre_existing OR policy_tenure_months >= ped_waiting_months)   THEN 'PED_WAITING'
              WHEN (plan_copay_pct + CASE WHEN is_network_at_admission THEN 0 ELSE 20 END) > 0
                   OR (CASE WHEN room_rent_per_day <= room_rent_cap THEN 0.0
                            ELSE round((1 - room_rent_cap / room_rent_per_day) * 100, 2) END) > 0
                   THEN 'PAYABLE_WITH_DEDUCTION'
              ELSE 'PAYABLE_IN_FULL' END                                                    AS reason_code
  FROM probe_claims
)
""")

spark.sql("SELECT * FROM claims.genai.adjudication_verifier ORDER BY claim_id").show(25, False)

In [0]:
# Step 3 - batch LLM adjudication in SQL: ai_query over the probe claims.
# Fixed JSON reply shape, temperature 0, reasoning kept, worked examples on MADE-UP claims only.
# The prompt carries claim FACTS ONLY - the verifier's answers are never shown to the LLM.

PROMPT_HEADER = """You are the first-pass adjudication engine for Saffron Shield health insurance.
Adjudicate each inpatient claim strictly by the rules below, then reply with ONLY a json object (no other text).

Claim facts given: admission_date, policy_start_date (first policy start), days_from_policy_start (calendar days, already computed), tenure_months (completed months from first policy start at admission, already computed), is_pre_existing, plan (plan_code, initial_waiting_days, ped_waiting_months, copay_pct, room_rent_cap_per_day), room_rent_per_day, network_at_admission.

Rules:
1. The initial waiting period passes if days_from_policy_start >= plan.initial_waiting_days.
2. The PED (pre-existing disease) waiting period rule has two cases. CASE A: if is_pre_existing is false, the PED waiting period PASSES immediately, no matter what tenure_months is. CASE B: only if is_pre_existing is true, the PED waiting period passes if tenure_months >= plan.ped_waiting_months.
3. waiting_period_ok = both waiting rules pass. payable = waiting_period_ok (no exclusions apply).
4. network_at_admission = the given network flag.
5. room_rent_deduction_pct = 0.0 if room_rent_per_day <= plan.room_rent_cap_per_day, else round((1 - cap/rent) * 100, 2) with cap = plan.room_rent_cap_per_day and rent = room_rent_per_day. Compute the percentage to exactly 2 decimals.
6. copay_pct = plan.copay_pct, plus 20 if network_at_admission is false.
7. reason_code, first matching rule: "INITIAL_WAITING" if rule 1 fails; else "PED_WAITING" if rule 2 fails; else "PAYABLE_WITH_DEDUCTION" if payable and (room_rent_deduction_pct > 0 or copay_pct > 0); else "PAYABLE_IN_FULL".

Work through every rule explicitly in the "rule_checks" object, restating the input values you compared, then give the decision. Reply with exactly this json shape:
{"rule_checks": {"days_from_policy_start": <integer>, "initial_waiting_met": <true|false>, "is_pre_existing": <true|false>, "tenure_months": <integer>, "ped_waiting_months": <integer>, "ped_waiting_met": <true|false>, "rent_exceeds_cap": <true|false>, "deduction_pct_computed": <number>, "copay_pct_computed": <integer>}, "payable": <true|false>, "waiting_period_ok": <true|false>, "network_at_admission": <true|false>, "room_rent_deduction_pct": <number with 2 decimals>, "copay_pct": <integer>, "reason_code": "<INITIAL_WAITING|PED_WAITING|PAYABLE_WITH_DEDUCTION|PAYABLE_IN_FULL>", "reasoning": "<one or two sentences citing the rule you applied>"}

Worked examples (fictional claims, for shape only):

CLAIM NOT-REAL-1 facts: {"admission_date":"2026-01-15","policy_start_date":"2026-01-01","days_from_policy_start":14,"tenure_months":0,"is_pre_existing":false,"plan":{"plan_code":"SILVER_IND","initial_waiting_days":30,"ped_waiting_months":36,"copay_pct":0,"room_rent_cap_per_day":"3000.0"},"room_rent_per_day":2000,"network_at_admission":true}
Reply: {"rule_checks": {"days_from_policy_start": 14, "initial_waiting_met": false, "is_pre_existing": false, "tenure_months": 0, "ped_waiting_months": 36, "ped_waiting_met": true, "rent_exceeds_cap": false, "deduction_pct_computed": 0.0, "copay_pct_computed": 0}, "payable": false, "waiting_period_ok": false, "network_at_admission": true, "room_rent_deduction_pct": 0.0, "copay_pct": 0, "reason_code": "INITIAL_WAITING", "reasoning": "Only 14 days from policy start, below the 30-day initial waiting period, so the claim is not payable."}

CLAIM NOT-REAL-2 facts: {"admission_date":"2025-06-01","policy_start_date":"2020-01-01","days_from_policy_start":1979,"tenure_months":65,"is_pre_existing":true,"plan":{"plan_code":"GOLD_IND","initial_waiting_days":30,"ped_waiting_months":36,"copay_pct":0,"room_rent_cap_per_day":"6000.0"},"room_rent_per_day":7000,"network_at_admission":true}
Reply: {"rule_checks": {"days_from_policy_start": 1979, "initial_waiting_met": true, "is_pre_existing": true, "tenure_months": 65, "ped_waiting_months": 36, "ped_waiting_met": true, "rent_exceeds_cap": true, "deduction_pct_computed": 14.29, "copay_pct_computed": 0}, "payable": true, "waiting_period_ok": true, "network_at_admission": true, "room_rent_deduction_pct": 14.29, "copay_pct": 0, "reason_code": "PAYABLE_WITH_DEDUCTION", "reasoning": "The condition is pre-existing and tenure 65 months clears the 36-month PED wait; rent 7000 exceeds the 6000 cap, so a 14.29% room rent deduction applies."}

CLAIM NOT-REAL-3 facts: {"admission_date":"2026-02-10","policy_start_date":"2024-06-01","days_from_policy_start":619,"tenure_months":20,"is_pre_existing":true,"plan":{"plan_code":"CORP_GROUP","initial_waiting_days":0,"ped_waiting_months":12,"copay_pct":0,"room_rent_cap_per_day":"5000.0"},"room_rent_per_day":4000,"network_at_admission":true}
Reply: {"rule_checks": {"days_from_policy_start": 619, "initial_waiting_met": true, "is_pre_existing": true, "tenure_months": 20, "ped_waiting_months": 12, "ped_waiting_met": true, "rent_exceeds_cap": false, "deduction_pct_computed": 0.0, "copay_pct_computed": 0}, "payable": true, "waiting_period_ok": true, "network_at_admission": true, "room_rent_deduction_pct": 0.0, "copay_pct": 0, "reason_code": "PAYABLE_IN_FULL", "reasoning": "The condition is pre-existing but tenure 20 months clears the 12-month group PED wait, and rent 4000 is within the 5000 cap, so the claim is payable in full."}

"""
assert "'" not in PROMPT_HEADER, "keep the prompt free of single quotes (SQL literal)"

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW llm_input AS
SELECT claim_id,
  to_json(named_struct(
    'admission_date', admission_date,
    'policy_start_date', CAST(policy_start_date AS STRING),
    'days_from_policy_start', datediff(admission_date, policy_start_date),
    'tenure_months', policy_tenure_months,
    'is_pre_existing', is_pre_existing,
    'plan', named_struct(
      'plan_code', plan_code,
      'initial_waiting_days', initial_waiting_days,
      'ped_waiting_months', ped_waiting_months,
      'copay_pct', plan_copay_pct,
      'room_rent_cap_per_day', CAST(room_rent_cap AS STRING)),
    'room_rent_per_day', room_rent_per_day,
    'network_at_admission', is_network_at_admission)) AS facts
FROM probe_claims
""")

llm_reply_schema = '''payable BOOLEAN, waiting_period_ok BOOLEAN, network_at_admission BOOLEAN,
room_rent_deduction_pct DOUBLE, copay_pct INT, reason_code STRING, reasoning STRING'''

spark.sql(f"""
CREATE OR REPLACE TABLE claims.genai.adjudications AS
SELECT
  CAST(claim_id AS STRING)                  AS claim_id,
  CAST(j.payable AS BOOLEAN)                AS payable,
  CAST(j.waiting_period_ok AS BOOLEAN)      AS waiting_period_ok,
  CAST(j.network_at_admission AS BOOLEAN)   AS network_at_admission,
  CAST(round(j.room_rent_deduction_pct, 2) AS DOUBLE) AS room_rent_deduction_pct,
  CAST(j.copay_pct AS INT)                  AS copay_pct,
  CAST(j.reason_code AS STRING)             AS reason_code,
  CAST(j.reasoning AS STRING)               AS reasoning
FROM (
  SELECT claim_id,
         from_json(resp.result, '{llm_reply_schema}') AS j,
         resp.errorMessage AS err
  FROM (
    SELECT claim_id, ai_query(
      'databricks-meta-llama-3-3-70b-instruct',
      request => concat('{PROMPT_HEADER}', 'Now adjudicate this claim.\\nCLAIM ', claim_id, ' facts: ', facts, '\\nReply with only the json object.'),
      modelParameters => named_struct('temperature', 0.0, 'max_tokens', 700),
      responseFormat => '{{"type": "json_object"}}',
      failOnError => false
    ) AS resp
    FROM llm_input
  )
)
""")

bad = spark.sql("SELECT count(*) c FROM claims.genai.adjudications WHERE payable IS NULL OR reasoning IS NULL").collect()[0].c
assert bad == 0, f"{bad} rows failed to parse - inspect err above"
print(spark.sql("SELECT * FROM claims.genai.adjudications ORDER BY claim_id").toPandas().to_string())

In [0]:
# Step 4 - agreement: LLM vs verifier, per the Definitions sheet.
# Agree = payable, waiting_period_ok, network_at_admission, copay_pct, reason_code equal
# AND room_rent_deduction_pct within 1.0 point. Trap claims must all agree.

spark.sql("""
CREATE OR REPLACE TEMP VIEW agreement AS
SELECT v.claim_id,
       v.payable = a.payable
       AND v.waiting_period_ok = a.waiting_period_ok
       AND v.network_at_admission = a.network_at_admission
       AND v.copay_pct = a.copay_pct
       AND v.reason_code = a.reason_code
       AND abs(v.room_rent_deduction_pct - a.room_rent_deduction_pct) <= 1.0 AS agree
FROM claims.genai.adjudication_verifier v
JOIN claims.genai.adjudications a ON a.claim_id = v.claim_id
""")

n, agree_n = spark.sql("SELECT count(*) n, sum(CASE WHEN agree THEN 1 ELSE 0 END) a FROM agreement").collect()[0]
rate = agree_n / n
print(f"agreement: {agree_n}/{n} = {rate:.2%}  (threshold >= 90%)")

traps = ["CLM-20001", "CLM-20002", "CLM-20003", "CLM-20004"]
trap_rows = spark.sql(f"SELECT * FROM agreement WHERE claim_id IN ('{",".join(traps)}') ORDER BY claim_id").collect()
traps_ok = all(r.agree for r in trap_rows)
for r in trap_rows:
    print("trap", r.claim_id, "agree =", r.agree)

assert rate >= 0.90 and traps_ok, "agreement threshold or trap-claim check FAILED"
print("PASS - LLM agrees with the verifier on >= 90% of claims and on every trap claim")

# where they disagree (if ever)
spark.sql("""
SELECT v.claim_id, v.reason_code AS verifier_rc, a.reason_code AS llm_rc,
       v.room_rent_deduction_pct AS v_ded, a.room_rent_deduction_pct AS a_ded,
       v.copay_pct AS v_copay, a.copay_pct AS a_copay
FROM claims.genai.adjudication_verifier v
JOIN claims.genai.adjudications a ON a.claim_id = v.claim_id
WHERE a.claim_id NOT IN (SELECT claim_id FROM agreement WHERE agree)
ORDER BY v.claim_id
""").show()


In [0]:
# Step 5 - independent verification of the R8 deliverables.

import datetime

cols = {r.claim_id: r for r in spark.sql("""
  SELECT e.claim_id, e.admission_date, e.policy_start_date, e.policy_tenure_months,
         e.is_pre_existing, e.room_rent_per_day, e.is_network_at_admission,
         pl.initial_waiting_days, pl.ped_waiting_months, pl.copay_pct, pl.room_rent_cap_per_day
  FROM claims.gold.claims_enriched e JOIN claims.raw.plans pl ON pl.plan_code = e.plan_code_at_admission
  WHERE e.claim_id IN ('CLM-11531', 'CLM-20004')
""").collect()}

def hand_adjudicate(c):
    adm = datetime.date.fromisoformat(c.admission_date)
    initial_ok = (adm - c.policy_start_date).days >= int(c.initial_waiting_days)
    ped_ok = (not c.is_pre_existing) or c.policy_tenure_months >= int(c.ped_waiting_months)
    wpo = initial_ok and ped_ok
    cap, rent = float(c.room_rent_cap_per_day), float(c.room_rent_per_day)
    ded = 0.0 if rent <= cap else round((1 - cap / rent) * 100, 2)
    copay = int(c.copay_pct) + (0 if c.is_network_at_admission else 20)
    rc = ("INITIAL_WAITING" if not initial_ok else "PED_WAITING" if not ped_ok
          else "PAYABLE_WITH_DEDUCTION" if (ded > 0 or copay > 0) else "PAYABLE_IN_FULL")
    return wpo, wpo, c.is_network_at_admission, ded, copay, rc

for cid in ("CLM-11531", "CLM-20004"):
    exp = hand_adjudicate(cols[cid])
    v = spark.sql(f"SELECT * FROM claims.genai.adjudication_verifier WHERE claim_id = '{cid}'").collect()[0]
    got = (v.payable, v.waiting_period_ok, v.network_at_admission,
           round(v.room_rent_deduction_pct, 2), v.copay_pct, v.reason_code)
    print(cid, "hand:", exp, "| verifier:", got)
    assert exp == got, f"hand recompute mismatch on {cid}"

# table types exactly as specced
types = {r.col_name: r.data_type for r in spark.sql("DESCRIBE TABLE claims.genai.adjudication_verifier").collect()
         if r.col_name and not r.col_name.startswith("#")}
assert types == {"claim_id": "string", "payable": "boolean", "waiting_period_ok": "boolean",
                 "network_at_admission": "boolean", "room_rent_deduction_pct": "double",
                 "copay_pct": "int", "reason_code": "string"}, types
types_a = {r.col_name: r.data_type for r in spark.sql("DESCRIBE TABLE claims.genai.adjudications").collect()
           if r.col_name and not r.col_name.startswith("#")}
assert types_a == {**types, "reasoning": "string"}, types_a
assert spark.sql("SELECT count(*) c FROM claims.genai.adjudication_probes").collect()[0].c == 20

# reasoning rule: >= 18 rows carry 20 characters or more
long_reasoning = spark.sql("SELECT count(*) c FROM claims.genai.adjudications WHERE length(reasoning) >= 20").collect()[0].c
print("rows with reasoning >= 20 chars:", long_reasoning)
assert long_reasoning >= 18

# every LLM reply parsed to a valid typed row
n_null = spark.sql("""SELECT count(*) c FROM claims.genai.adjudications
                      WHERE payable IS NULL OR waiting_period_ok IS NULL OR network_at_admission IS NULL
                         OR room_rent_deduction_pct IS NULL OR copay_pct IS NULL OR reason_code IS NULL
                         OR reasoning IS NULL OR length(reasoning) = 0""").collect()[0].c
assert n_null == 0

print("PASS - hand recompute matches verifier; table types exact; 20 probe rows; LLM replies all valid JSON with reasoning")

In [0]:
%sql
SELECT * FROM claims.genai.adjudications ORDER BY claim_id;

![](![image_1794569694091.png](./image_1794569694091.png "image_1794569694091.png"))